In [2]:
# =========================
# RANDOM FOREST CLASSIFIER
# =========================

# 1. IMPORT LIBRARIES
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

# 2. LOAD DATA
df = pd.read_csv("../data/rf.csv")

# 3. DEFINE FEATURES AND TARGET
X = df.drop("target_class", axis=1)
y = df["target_class"]

# 4. SPLIT DATA INTO 80% TRAIN, 10% VALIDATION, 10% TEST

# First split: 80% train and 20% temp
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# Second split: split temp into 10% validation and 10% test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)

print("Shapes:")
print("X_train:", X_train.shape)
print("X_val  :", X_val.shape)
print("X_test :", X_test.shape)
print()



Shapes:
X_train: (8000, 6)
X_val  : (1000, 6)
X_test : (1000, 6)



In [3]:
# 4. IDENTIFY NUMERIC AND CATEGORICAL COLUMNS
numeric_features = X_train.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X_train.select_dtypes(include=["object", "category", "bool"]).columns.tolist()

print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)
print()

# 5. PREPROCESSING
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

# 6. PIPELINE
pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(random_state=42))
])

# 7. RANDOM SEARCH SPACE
param_distributions = {
    "model__n_estimators": [100, 150, 200, 300],
    "model__max_depth": [None, 5, 10, 15, 20],
    "model__min_samples_split": [2, 5, 10],
    "model__min_samples_leaf": [1, 2, 4],
    "model__max_features": ["sqrt", "log2"]
}

# 8. RANDOMIZED SEARCH
random_search = RandomizedSearchCV(
    estimator=pipeline,
    param_distributions=param_distributions,
    n_iter=20,              # only 20 random combinations
    scoring="accuracy",
    cv=10,
    verbose=2,
    random_state=42,
    n_jobs=1,               # safer for your computer
    refit=True
)

# 9. FIT ONLY ON TRAINING DATA
random_search.fit(X_train, y_train)

print("\nBest Parameters:")
print(random_search.best_params_)

print("\nBest Cross-Validation Accuracy:")
print(random_search.best_score_)

# 10. BEST MODEL
best_model = random_search.best_estimator_

# 11. EVALUATION FUNCTION
def evaluate_model(model, X_data, y_data, dataset_name):
    y_pred = model.predict(X_data)

    acc = accuracy_score(y_data, y_pred)
    cm = confusion_matrix(y_data, y_pred)
    report = classification_report(y_data, y_pred)

    print(f"\n{'='*50}")
    print(f"{dataset_name} RESULTS")
    print(f"{'='*50}")
    print("Accuracy:", round(acc, 4))
    print("\nConfusion Matrix:")
    print(cm)
    print("\nClassification Report:")
    print(report)

Numeric features: ['age', 'income_k', 'credit_score', 'tenure_years', 'monthly_spend', 'satisfaction_score']
Categorical features: []

Fitting 10 folds for each of 20 candidates, totalling 200 fits
[CV] END model__max_depth=15, model__max_features=sqrt, model__min_samples_leaf=1, model__min_samples_split=10, model__n_estimators=100; total time=   2.3s
[CV] END model__max_depth=15, model__max_features=sqrt, model__min_samples_leaf=1, model__min_samples_split=10, model__n_estimators=100; total time=   2.1s
[CV] END model__max_depth=15, model__max_features=sqrt, model__min_samples_leaf=1, model__min_samples_split=10, model__n_estimators=100; total time=   2.0s
[CV] END model__max_depth=15, model__max_features=sqrt, model__min_samples_leaf=1, model__min_samples_split=10, model__n_estimators=100; total time=   2.1s
[CV] END model__max_depth=15, model__max_features=sqrt, model__min_samples_leaf=1, model__min_samples_split=10, model__n_estimators=100; total time=   2.1s
[CV] END model__max_de

In [4]:
# 13. EVALUATE ON TRAINING SET
evaluate_model(best_model, X_train, y_train, "TRAINING SET")


TRAINING SET RESULTS
Accuracy: 0.8986

Confusion Matrix:
[[3618  382]
 [ 429 3571]]

Classification Report:
              precision    recall  f1-score   support

           0       0.89      0.90      0.90      4000
           1       0.90      0.89      0.90      4000

    accuracy                           0.90      8000
   macro avg       0.90      0.90      0.90      8000
weighted avg       0.90      0.90      0.90      8000



In [5]:

# 14. EVALUATE ON VALIDATION SET
evaluate_model(best_model, X_val, y_val, "VALIDATION SET")


VALIDATION SET RESULTS
Accuracy: 0.803

Confusion Matrix:
[[409  91]
 [106 394]]

Classification Report:
              precision    recall  f1-score   support

           0       0.79      0.82      0.81       500
           1       0.81      0.79      0.80       500

    accuracy                           0.80      1000
   macro avg       0.80      0.80      0.80      1000
weighted avg       0.80      0.80      0.80      1000



In [6]:
# 15. EVALUATE ON TEST SET
evaluate_model(best_model, X_test, y_test, "TEST SET")


TEST SET RESULTS
Accuracy: 0.801

Confusion Matrix:
[[397 103]
 [ 96 404]]

Classification Report:
              precision    recall  f1-score   support

           0       0.81      0.79      0.80       500
           1       0.80      0.81      0.80       500

    accuracy                           0.80      1000
   macro avg       0.80      0.80      0.80      1000
weighted avg       0.80      0.80      0.80      1000



In [7]:
# 16. FEATURE IMPORTANCE
# First get transformed feature names after preprocessing
feature_names = best_model.named_steps["preprocessor"].get_feature_names_out()

# Then get feature importances from Random Forest
importances = best_model.named_steps["model"].feature_importances_

feature_importance_df = pd.DataFrame({
    "feature": feature_names,
    "importance": importances
}).sort_values(by="importance", ascending=False)

print("\n" + "="*50)
print("FEATURE IMPORTANCE")
print("="*50)
print(feature_importance_df)

# Optional: print top 10 most important features
print("\nTop 10 Most Important Features:")
print(feature_importance_df.head(10))


FEATURE IMPORTANCE
                   feature  importance
2        num__credit_score    0.348528
5  num__satisfaction_score    0.211434
4       num__monthly_spend    0.170546
3        num__tenure_years    0.134623
1            num__income_k    0.085762
0                 num__age    0.049106

Top 10 Most Important Features:
                   feature  importance
2        num__credit_score    0.348528
5  num__satisfaction_score    0.211434
4       num__monthly_spend    0.170546
3        num__tenure_years    0.134623
1            num__income_k    0.085762
0                 num__age    0.049106
